# Results for model: mistralai/mistral-7b-instruct-v0.2

In [1]:
import polars as pl
import xgboost as xgb

# Load 'train.parquet' using Polars
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering: Calculate a global TOP_QUANTILE of 'feature_00'
# relative to 'responder_6' across rolling batches of 'date_id'
window_size = int(0.15 * len(df))
window = pl.Window.rolling(size=window_size, stride=1)

top_quantile = df.select([pl.col('feature_00'), pl.col('responder_6'), pl.col('date_id').alias('window_id')]) \
              .with_columns(top=pl.col('feature_00').quantile(0.875).alias('top_quantile_feature_00')) \
              .groupby(['window_id']).agg(pl.col('top_quantile_feature_00').last().alias('top_quantile')) \
              .drop(['window_id', 'top_quantile_feature_00']) \
              .rename({'feature_00': 'feature_00_top_quantile', 'responder_6': 'target'})

# Merge the original dataframe with the feature engineered data
df_engineered = df.join(top_quantile, on=['date_id'], how='outer')

# Train an XGBoost Regressor on the target 'responder_6'
X = df_engineered.select(['feature_00_top_quantile'])
y = df_engineered.select(['target'])

# Fitting the model
xgb_reg = xgb.XGBRegressor(objective="reg:squarederror", boosting_type="gblinear", maximize_output=False)
xgb_reg.fit(X.to_pandas(), y.to_pandas())

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet